In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
import os, sys
real = '/content/drive/MyDrive/0000_Kaan_CDS'
link = '/content/drive/MyDrive/0000_Kaan_CDS'
if os.path.exists(real) and not os.path.exists(link):
    os.symlink(real, link)

src_dir = '/content/drive/MyDrive/0000_Kaan_CDS/src'
os.chdir(src_dir)
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from cds_config import *
from m1_calibration import load_calibrated_probs

print('SCENARIOS:', 'SCENARIOS' in dir())
print('PATHS:', 'PATHS' in dir())
print('load_calibrated_probs:', 'load_calibrated_probs' in dir())

In [ ]:
# ── Load calibrated probabilities ──
data = {}
for scenario in SCENARIOS:
    for stage in [1, 2]:
        for split in ['oof', 'test']:
            key = f"{scenario}_S{stage}_{split}"
            data[key] = load_calibrated_probs(PATHS, stage, scenario, split)
            print(f"  {key}: {data[key].shape}")

# ── Column check ──
print("\n─── Stage 1 calibrated columns ───")
print([c for c in data['CBC_Only_S1_oof'].columns if 'cal' in c.lower()])

print("\n─── Stage 2 calibrated columns ───")
print([c for c in data['CBC_Only_S2_oof'].columns if 'cal' in c.lower()])

# ── Stage 1: prob_IAS_cal distribution summary ──
print("\n═══ Stage 1 — prob_IAS_cal Distribution ═══")
for scenario in SCENARIOS:
    df = data[f'{scenario}_S1_oof']
    print(f"\n{scenario}:")
    print(df['prob_IAS_cal'].describe().round(4).to_string())

# ── Stage 2: argmax confidence distribution summary ──
S2_CAL_COLS = ['prob_DEA_cal', 'prob_HA_cal', 'prob_HGB_HTZ_cal', 'prob_Normal_cal']
print("\n═══ Stage 2 — max(prob_*_cal) Distribution ═══")
for scenario in SCENARIOS:
    df = data[f'{scenario}_S2_oof']
    conf = df[S2_CAL_COLS].max(axis=1)
    print(f"\n{scenario}:")
    print(conf.describe().round(4).to_string())

## Cell 2 — 2A: Stage 1 Threshold Optimization

In [ ]:
# ============================================================
# Cell 2 — 2A: Stage 1 Threshold Optimization
# ============================================================
import numpy as np
import pandas as pd
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             accuracy_score)

def stage1_threshold_sweep(df, thresholds=np.arange(0.05, 0.96, 0.01)):
    """Stage 1 threshold sweep. target=1 → IAS (positive class)."""
    y_true = df['target'].values
    probs = df['prob_IAS_cal'].values

    records = []
    for t in thresholds:
        y_pred = (probs >= t).astype(int)
        if y_pred.sum() == 0 or y_pred.sum() == len(y_pred):
            continue

        sens = recall_score(y_true, y_pred, pos_label=1)
        spec = recall_score(y_true, y_pred, pos_label=0)
        f1   = f1_score(y_true, y_pred, pos_label=1)
        prec = precision_score(y_true, y_pred, pos_label=1)
        acc  = accuracy_score(y_true, y_pred)
        youden = sens + spec - 1

        records.append({
            'threshold': round(t, 2),
            'sensitivity': sens, 'specificity': spec,
            'precision': prec, 'f1': f1,
            'accuracy': acc, 'youden_j': youden,
            'n_pos_pred': int(y_pred.sum()),
            'n_neg_pred': int(len(y_pred) - y_pred.sum())
        })

    return pd.DataFrame(records)

# ── Sweep per scenario (OOF) ──
sweep_results = {}
for scenario in SCENARIOS:
    df_oof = data[f'{scenario}_S1_oof']
    sweep = stage1_threshold_sweep(df_oof)
    sweep_results[scenario] = sweep

    f1_opt = sweep.loc[sweep['f1'].idxmax()]
    youden_opt = sweep.loc[sweep['youden_j'].idxmax()]

    clinical = sweep[sweep['sensitivity'] >= 0.95]
    if len(clinical) > 0:
        clinical_opt = clinical.loc[clinical['specificity'].idxmax()]
    else:
        clinical_opt = sweep.loc[sweep['sensitivity'].idxmax()]
        print(f"  ⚠ {scenario}: Sens≥0.95 not achievable, highest sens used")

    thesis_t = 0.53 if scenario == 'CBC_Only' else 0.44
    thesis_row = sweep.iloc[(sweep['threshold'] - thesis_t).abs().argsort()[:1]].iloc[0]

    print(f"\n{'═'*65}")
    print(f"  {scenario} — Stage 1 Threshold Optimization (OOF, n={len(df_oof)})")
    print(f"{'═'*65}")
    print(f"  {'Criterion':<18} {'Thresh':>7} {'Sens':>7} {'Spec':>7} {'F1':>7} {'Acc':>7} {'Youden':>7}")
    print(f"  {'─'*58}")
    print(f"  {'F1-optimal':<18} {f1_opt['threshold']:>7.2f} {f1_opt['sensitivity']:>7.3f} {f1_opt['specificity']:>7.3f} {f1_opt['f1']:>7.3f} {f1_opt['accuracy']:>7.3f} {f1_opt['youden_j']:>7.3f}")
    print(f"  {'Youden-J':<18} {youden_opt['threshold']:>7.2f} {youden_opt['sensitivity']:>7.3f} {youden_opt['specificity']:>7.3f} {youden_opt['f1']:>7.3f} {youden_opt['accuracy']:>7.3f} {youden_opt['youden_j']:>7.3f}")
    print(f"  {'Clinical (S≥.95)':<18} {clinical_opt['threshold']:>7.2f} {clinical_opt['sensitivity']:>7.3f} {clinical_opt['specificity']:>7.3f} {clinical_opt['f1']:>7.3f} {clinical_opt['accuracy']:>7.3f} {clinical_opt['youden_j']:>7.3f}")
    print(f"  {'Thesis (reference)':<18} {thesis_row['threshold']:>7.2f} {thesis_row['sensitivity']:>7.3f} {thesis_row['specificity']:>7.3f} {thesis_row['f1']:>7.3f} {thesis_row['accuracy']:>7.3f} {thesis_row['youden_j']:>7.3f}")

## Cell 3 Figure: Stage 1 Threshold Curves (1×2 panel)

In [ ]:
# ============================================================
# Cell 3 — Figure: Stage 1 Threshold Curves (1×2 panel)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

for idx, scenario in enumerate(SCENARIOS):
    ax = axes[idx]
    sweep = sweep_results[scenario]

    # ── Metric curves ──
    ax.plot(sweep['threshold'], sweep['sensitivity'],
            color=PALETTE['highlight'], lw=2, label='Sensitivity')
    ax.plot(sweep['threshold'], sweep['specificity'],
            color=PALETTE['base1'], lw=2, label='Specificity')
    ax.plot(sweep['threshold'], sweep['f1'],
            color=PALETTE['accent3'], lw=2, label='F1 Score')
    ax.plot(sweep['threshold'], sweep['youden_j'],
            color=PALETTE['accent2'], lw=1.5, ls='--', label="Youden's J")

    # ── Mark optimal points ──
    f1_row = sweep.loc[sweep['f1'].idxmax()]
    youden_row = sweep.loc[sweep['youden_j'].idxmax()]
    clinical = sweep[sweep['sensitivity'] >= 0.95]
    clinical_row = clinical.loc[clinical['specificity'].idxmax()] if len(clinical) > 0 else sweep.loc[sweep['sensitivity'].idxmax()]

    ax.axvline(f1_row['threshold'], color=PALETTE['accent3'], ls=':', alpha=0.6, lw=1)
    ax.axvline(youden_row['threshold'], color=PALETTE['accent2'], ls=':', alpha=0.6, lw=1)
    ax.axvline(clinical_row['threshold'], color=PALETTE['highlight'], ls=':', alpha=0.6, lw=1)

    # Thesis reference
    thesis_t = 0.53 if scenario == 'CBC_Only' else 0.44
    ax.axvline(thesis_t, color=PALETTE['base2'], ls='--', lw=1.5, label=f'Thesis ({thesis_t})')

    # ── Annotations ──
    y_offset = 0.03
    ax.annotate(f"F1-opt\n({f1_row['threshold']:.2f})",
                xy=(f1_row['threshold'], f1_row['f1']),
                fontsize=7, ha='center', color=PALETTE['accent3'])
    ax.annotate(f"Youden\n({youden_row['threshold']:.2f})",
                xy=(youden_row['threshold'], youden_row['youden_j']),
                fontsize=7, ha='center', color=PALETTE['accent2'])

    # ── Histogram (secondary y-axis) ──
    ax2 = ax.twinx()
    df_oof = data[f'{scenario}_S1_oof']
    ax2.hist(df_oof['prob_IAS_cal'], bins=50, alpha=0.12, color=PALETTE['base1'], zorder=0)
    ax2.set_ylabel('Frequency', fontsize=9, color=PALETTE['base1'])
    ax2.tick_params(axis='y', labelcolor=PALETTE['base1'], labelsize=8)

    # ── Formatting ──
    ax.set_xlabel('Decision threshold', fontsize=10)
    ax.set_ylabel('Metric value', fontsize=10)
    ax.set_title(scenario, fontsize=11, fontweight='bold')
    ax.set_xlim(0.05, 0.95)
    ax.set_ylim(-0.1, 1.05)
    ax.legend(loc='lower left', fontsize=7, frameon=False)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(0.1))

for ax in axes:
    despine_all(ax)
fig.suptitle('Stage 1 — Threshold Optimization Curves', fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()

save_fig(fig, PATHS.module_dir('m2_thresholds', 'plots'), 's1_threshold_curves')
plt.show()

## Cell 4 — Stage 1: Test Set Validation + Candidates Table

In [ ]:
# ============================================================
# Cell 4 — Stage 1: Test Set Validation + Candidates Table
# ============================================================

def evaluate_threshold(df, threshold):
    """Compute Stage 1 metrics at a single threshold."""
    y_true = df['target'].values
    y_pred = (df['prob_IAS_cal'].values >= threshold).astype(int)

    if y_pred.sum() == 0 or y_pred.sum() == len(y_pred):
        return None

    return {
        'sensitivity': recall_score(y_true, y_pred, pos_label=1),
        'specificity': recall_score(y_true, y_pred, pos_label=0),
        'precision':   precision_score(y_true, y_pred, pos_label=1),
        'f1':          f1_score(y_true, y_pred, pos_label=1),
        'accuracy':    accuracy_score(y_true, y_pred),
        'youden_j':    recall_score(y_true, y_pred, pos_label=1) + recall_score(y_true, y_pred, pos_label=0) - 1
    }

# ── Collect all candidates ──
candidates = []
for scenario in SCENARIOS:
    sweep = sweep_results[scenario]
    f1_row = sweep.loc[sweep['f1'].idxmax()]
    youden_row = sweep.loc[sweep['youden_j'].idxmax()]
    clinical = sweep[sweep['sensitivity'] >= 0.95]
    clinical_row = clinical.loc[clinical['specificity'].idxmax()] if len(clinical) > 0 else sweep.loc[sweep['sensitivity'].idxmax()]
    thesis_t = 0.53 if scenario == 'CBC_Only' else 0.44

    candidate_thresholds = {
        'F1-optimal':        f1_row['threshold'],
        'Youden-J':          youden_row['threshold'],
        'Clinical (S≥.95)':  clinical_row['threshold'],
        'Thesis (reference)': thesis_t
    }

    for method, t in candidate_thresholds.items():
        for split in ['oof', 'test']:
            df_eval = data[f'{scenario}_S1_{split}']
            metrics = evaluate_threshold(df_eval, t)
            if metrics:
                candidates.append({
                    'scenario': scenario,
                    'method': method,
                    'threshold': t,
                    'split': split,
                    **metrics
                })

df_candidates = pd.DataFrame(candidates)

# ── Show table: OOF vs Test comparison ──
for scenario in SCENARIOS:
    print(f"\n{'═'*75}")
    print(f"  {scenario} — Stage 1 Threshold Candidates (OOF vs Test)")
    print(f"{'═'*75}")
    print(f"  {'Method':<18} {'Thr':>5} {'Split':<5} {'Sens':>7} {'Spec':>7} {'F1':>7} {'Acc':>7} {'Youden':>7}")
    print(f"  {'─'*68}")

    sub = df_candidates[df_candidates['scenario'] == scenario]
    for method in ['F1-optimal', 'Youden-J', 'Clinical (S≥.95)', 'Thesis (reference)']:
        for split in ['oof', 'test']:
            row = sub[(sub['method'] == method) & (sub['split'] == split)]
            if len(row) == 0:
                continue
            r = row.iloc[0]
            marker = '←' if split == 'test' else ''
            print(f"  {method:<18} {r['threshold']:>5.2f} {split:<5} {r['sensitivity']:>7.3f} {r['specificity']:>7.3f} {r['f1']:>7.3f} {r['accuracy']:>7.3f} {r['youden_j']:>7.3f} {marker}")
        print()

# ── Save to Excel ──
out_path = PATHS.module_dir('m2_thresholds', 'metrics') / 'threshold_candidates.xlsx'
df_candidates.round(4).to_excel(out_path, index=False)
print(f"\n✓ Saved: {out_path}")

## Cell 5 — 2B: Stage 2 Confidence Zones (Accuracy–Coverage)

In [ ]:
# ============================================================
# Cell 5 — 2B: Stage 2 Confidence Zone Sweep
# ============================================================
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

S2_CAL_COLS = ['prob_DEA_cal', 'prob_HA_cal', 'prob_HGB_HTZ_cal', 'prob_Normal_cal']
S2_CLASSES  = ['DEA', 'HA', 'HGB_HTZ', 'Normal']

def stage2_confidence_sweep(df, thresholds=np.arange(0.25, 0.96, 0.01)):
    """
    Stage 2 confidence threshold sweep.
    argmax confidence < threshold → 'uncertain'.
    Coverage = fraction confident, Accuracy = accuracy among confident.
    """
    probs = df[S2_CAL_COLS].values
    y_pred_class = np.argmax(probs, axis=1)
    confidence = np.max(probs, axis=1)
    y_true = df['target'].values

    records = []
    for t in thresholds:
        mask = confidence >= t
        coverage = mask.mean()
        if mask.sum() == 0:
            continue

        acc_covered = accuracy_score(y_true[mask], y_pred_class[mask])

        per_class = {}
        for i, cls in enumerate(S2_CLASSES):
            cls_mask = mask & (y_true == i)
            if cls_mask.sum() > 0:
                per_class[f'acc_{cls}'] = (y_pred_class[cls_mask] == y_true[cls_mask]).mean()
                per_class[f'n_{cls}'] = int(cls_mask.sum())
            else:
                per_class[f'acc_{cls}'] = np.nan
                per_class[f'n_{cls}'] = 0

        records.append({
            'threshold': round(t, 2),
            'coverage': coverage,
            'accuracy': acc_covered,
            'n_covered': int(mask.sum()),
            'n_uncertain': int((~mask).sum()),
            **per_class
        })

    return pd.DataFrame(records)

# ── Target encoding verification ──
print("═══ Stage 2 Target Encoding Check ═══")
for scenario in SCENARIOS:
    df = data[f'{scenario}_S2_oof']
    print(f"\n{scenario}: target dtype={df['target'].dtype}, unique={sorted(df['target'].unique())}")
    for i, cls in enumerate(S2_CLASSES):
        mask = df['target'] == i
        if mask.sum() > 0:
            mean_prob = df.loc[mask, S2_CAL_COLS[i]].mean()
            print(f"  target={i} → mean {S2_CAL_COLS[i]}: {mean_prob:.3f} (n={mask.sum()}) → {cls}")

# ── Sweep for both scenarios (OOF) ──
conf_sweep = {}
for scenario in SCENARIOS:
    df_oof = data[f'{scenario}_S2_oof']
    conf_sweep[scenario] = stage2_confidence_sweep(df_oof)

# ── Result table ──
for scenario in SCENARIOS:
    sweep = conf_sweep[scenario]
    df_oof = data[f'{scenario}_S2_oof']

    print(f"\n{'═'*65}")
    print(f"  {scenario} — Stage 2 Confidence Sweep (OOF, n={len(df_oof)})")
    print(f"{'═'*65}")
    print(f"  {'Thresh':>7} {'Cover':>8} {'Acc':>8} {'n_cov':>7} {'n_unc':>7}")
    print(f"  {'─'*42}")

    for t in [0.30, 0.35, 0.40, 0.50, 0.60, 0.70, 0.80]:
        row = sweep[sweep['threshold'] == t]
        if len(row) > 0:
            r = row.iloc[0]
            print(f"  {r['threshold']:>7.2f} {r['coverage']:>8.3f} {r['accuracy']:>8.3f} {int(r['n_covered']):>7d} {int(r['n_uncertain']):>7d}")

## Cell 6 Stage 2: 3-Zone Definition + Accuracy-Coverage Plot

In [ ]:
# ============================================================
# Cell 6 — Stage 2: 3-Zone Definition + Accuracy-Coverage Plot
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Determine zone boundaries ──
# Strategy: HIGH zone → lowest threshold with accuracy ≥ 0.85
#           LOW zone  → point with minimal coverage loss (~0.35)

zone_configs = {}
for scenario in SCENARIOS:
    sweep = conf_sweep[scenario]

    # HIGH: lowest threshold achieving accuracy ≥ 0.85
    high_cands = sweep[sweep['accuracy'] >= 0.85]
    if len(high_cands) > 0:
        high_t = high_cands['threshold'].min()
    else:
        high_t = sweep.loc[sweep['accuracy'].idxmax(), 'threshold']

    # LOW: 0.35 (minimal coverage loss in both scenarios)
    low_t = 0.35

    zone_configs[scenario] = {'low': low_t, 'high': high_t}

    # Get metrics at the zone boundaries from the sweep
    low_row = sweep[sweep['threshold'] == low_t]
    high_row = sweep[sweep['threshold'] == high_t]

    print(f"{scenario}:")
    print(f"  LOW  (< {low_t}):  uncertain → manual review")
    if len(low_row) > 0:
        r = low_row.iloc[0]
        print(f"    → MED+HIGH coverage={r['coverage']:.3f}, acc={r['accuracy']:.3f}")
    print(f"  HIGH (≥ {high_t}): automated report")
    if len(high_row) > 0:
        r = high_row.iloc[0]
        print(f"    → HIGH coverage={r['coverage']:.3f}, acc={r['accuracy']:.3f}")
    print()

# ── Figure: Accuracy–Coverage + Zone boundaries (1×2 panel) ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, scenario in enumerate(SCENARIOS):
    ax = axes[idx]
    sweep = conf_sweep[scenario]
    zc = zone_configs[scenario]

    # Curves
    ax.plot(sweep['threshold'], sweep['accuracy'],
            color=PALETTE['highlight'], lw=2, label='Accuracy (covered)')
    ax.plot(sweep['threshold'], sweep['coverage'],
            color=PALETTE['accent1'], lw=2, label='Coverage')

    # Zone regions
    ax.axvspan(0.25, zc['low'], alpha=0.10, color=PALETTE['highlight'])
    ax.axvspan(zc['low'], zc['high'], alpha=0.08, color=PALETTE['accent2'])
    ax.axvspan(zc['high'], 0.96, alpha=0.10, color=PALETTE['accent1'])

    # Zone boundary lines
    ax.axvline(zc['low'], color=PALETTE['accent2'], ls='--', lw=1.5, alpha=0.8)
    ax.axvline(zc['high'], color=PALETTE['accent1'], ls='--', lw=1.5, alpha=0.8)

    # Zone labels (top)
    mid_low = (0.25 + zc['low']) / 2
    mid_med = (zc['low'] + zc['high']) / 2
    mid_high = (zc['high'] + 0.96) / 2
    ax.text(mid_low, 1.02, 'LOW', ha='center', fontsize=8, color=PALETTE['highlight'], fontweight='bold')
    ax.text(mid_med, 1.02, 'MEDIUM', ha='center', fontsize=8, color=PALETTE['accent2'], fontweight='bold')
    ax.text(mid_high, 1.02, 'HIGH', ha='center', fontsize=8, color=PALETTE['accent1'], fontweight='bold')

    # Thesis reference
    ax.axvline(0.30, color=PALETTE['base2'], ls=':', lw=1, label='Thesis (0.30)')

    ax.set_xlabel('Confidence Threshold', fontsize=10)
    ax.set_ylabel('Metric value', fontsize=10)
    ax.set_title(scenario, fontsize=11, fontweight='bold')
    ax.set_xlim(0.25, 0.95)
    ax.set_ylim(0.3, 1.05)
    ax.legend(loc='center left', fontsize=7, frameon=False)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(0.1))

for ax in axes:
    despine_all(ax)

fig.suptitle('Stage 2 — Confidence Zone Accuracy–Coverage', fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m2_thresholds', 'plots'), 's2_confidence_zones')
plt.show()

## Cell 7 — Per-Class Zone Distribution Analysis

In [ ]:
# ============================================================
# Cell 7 — Per-Class Zone Distribution Analysis
# ============================================================

def assign_zones(df, zone_config):
    """Assign a LOW/MEDIUM/HIGH zone to each sample."""
    probs = df[S2_CAL_COLS].values
    confidence = np.max(probs, axis=1)
    pred_class = np.argmax(probs, axis=1)

    zones = np.where(confidence < zone_config['low'], 'LOW',
            np.where(confidence >= zone_config['high'], 'HIGH', 'MEDIUM'))

    return pd.DataFrame({
        'target': df['target'].values,
        'pred_class': pred_class,
        'confidence': confidence,
        'zone': zones,
        'correct': (pred_class == df['target'].values).astype(int)
    })

# ── Display label map (internal DEA/HGB_HTZ → publication IDA/HGB HTZ) ──
_DISP = {'DEA': 'IDA', 'HGB_HTZ': 'HGB HTZ'}

# ── Zone assignment + distribution table ──
zone_dfs = {}
all_zone_stats = []

for scenario in SCENARIOS:
    df_oof = data[f'{scenario}_S2_oof']
    zc = zone_configs[scenario]
    zdf = assign_zones(df_oof, zc)
    zone_dfs[scenario] = zdf

    print(f"\n{'═'*70}")
    print(f"  {scenario} — Zone Distribution (OOF, n={len(zdf)})")
    print(f"  LOW < {zc['low']} ≤ MEDIUM < {zc['high']} ≤ HIGH")
    print(f"{'═'*70}")

    # Overall zone distribution
    print(f"\n  {'Zone':<8} {'N':>6} {'%':>7} {'Accuracy':>10}")
    print(f"  {'─'*34}")
    for zone in ['HIGH', 'MEDIUM', 'LOW']:
        mask = zdf['zone'] == zone
        n = mask.sum()
        pct = n / len(zdf) * 100
        acc = zdf.loc[mask, 'correct'].mean() if n > 0 else 0
        print(f"  {zone:<8} {n:>6} {pct:>6.1f}% {acc:>10.3f}")

    # Per-class × zone distribution
    print(f"\n  {'Class':<10} {'Zone':<8} {'N':>5} {'%_of_class':>11} {'Accuracy':>10}")
    print(f"  {'─'*48}")
    for i, cls in enumerate(S2_CLASSES):
        cls_mask = zdf['target'] == i
        cls_total = cls_mask.sum()
        for zone in ['HIGH', 'MEDIUM', 'LOW']:
            mask = cls_mask & (zdf['zone'] == zone)
            n = mask.sum()
            pct = n / cls_total * 100 if cls_total > 0 else 0
            acc = zdf.loc[mask, 'correct'].mean() if n > 0 else 0
            print(f"  {_DISP.get(cls, cls):<10} {zone:<8} {n:>5} {pct:>10.1f}% {acc:>10.3f}")

            all_zone_stats.append({
                'scenario': scenario, 'class': _DISP.get(cls, cls), 'zone': zone,
                'n': n, 'pct_of_class': round(pct, 1),
                'accuracy': round(acc, 3)
            })
        print()

# ── Save to Excel ──
df_zone_dist = pd.DataFrame(all_zone_stats)
out_path = PATHS.module_dir('m2_thresholds', 'metrics') / 'zone_distribution.xlsx'
df_zone_dist.to_excel(out_path, index=False)
print(f"✓ Saved: {out_path}")

## Cell 8 — Figure: Zone Distribution Stacked Bar (per-class)

In [ ]:
# ============================================================
# Cell 8 — Figure: Zone Distribution Stacked Bar (legend at bottom)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
zone_order = ['HIGH', 'MEDIUM', 'LOW']
zone_colors = {
    'HIGH':   PALETTE['accent1'],
    'MEDIUM': PALETTE['accent2'],
    'LOW':    PALETTE['highlight']
}

for idx, scenario in enumerate(SCENARIOS):
    ax = axes[idx]
    zdf = zone_dfs[scenario]

    x = np.arange(len(S2_CLASSES))
    width = 0.6
    bottom = np.zeros(len(S2_CLASSES))

    for zone in zone_order:
        pcts = []
        for i, cls in enumerate(S2_CLASSES):
            cls_total = (zdf['target'] == i).sum()
            zone_n = ((zdf['target'] == i) & (zdf['zone'] == zone)).sum()
            pcts.append(zone_n / cls_total * 100 if cls_total > 0 else 0)

        bars = ax.bar(x, pcts, width, bottom=bottom,
                       color=zone_colors[zone], label=zone if idx == 0 else '',
                       alpha=0.85, edgecolor='white', linewidth=0.5)

        for j, (p, b) in enumerate(zip(pcts, bottom)):
            if p > 5:
                ax.text(x[j], b + p/2, f'{p:.0f}%',
                        ha='center', va='center', fontsize=8, fontweight='bold',
                        color='white')

        bottom += pcts

    zc = zone_configs[scenario]
    ax.set_xticks(x)
    _disp = {'DEA': 'IDA', 'HGB_HTZ': 'HGB HTZ'}
    ax.set_xticklabels([_disp.get(c, c) for c in S2_CLASSES], fontsize=9)
    ax.set_ylabel('% of Class' if idx == 0 else '', fontsize=10)
    ax.set_title(f"{scenario}\n(LOW<{zc['low']} | MED<{zc['high']} | HIGH≥{zc['high']})",
                 fontsize=10, fontweight='bold')
    ax.set_ylim(0, 110)

for ax in axes:
    despine_all(ax)

# Legend below the figure, horizontal
fig.legend(['HIGH', 'MEDIUM', 'LOW'],
           loc='lower center', ncol=3, fontsize=9, frameon=False,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Stage 2 — Zone Distribution per Class', fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout(rect=[0, 0.05, 1, 1])
save_fig(fig, PATHS.module_dir('m2_thresholds', 'plots'), 's2_zone_distribution')
plt.show()

In [ ]:
# ── Cell 8b · Joint (cascade) sweep ───────────────────────────────────────────
# Sweeps the Stage 1 decision threshold and the Stage 2 confidence threshold
# jointly across the cascade. For each (s1_t, s2_t) pair:
#   coverage = (true-AAC passing S1 & S2_conf >= s2_t) / n_s1
#   accuracy = S2 classification accuracy of these "covered" samples
# The coverage denominator is n_s1 (total Stage 1 test), so coverage is reported
# relative to the full cohort entering the cascade.

def joint_sweep(df_s1, df_s2, s1_thresholds, s2_thresholds):
    s1_probs = df_s1['prob_IAS_cal'].values
    s1_true  = df_s1['target'].values            # 1=AAC(IAS), 0=OAC(DAS)  [internal codes]
    s2_probs = df_s2[S2_CAL_COLS].values
    s2_pred  = np.argmax(s2_probs, axis=1)
    s2_conf  = np.max(s2_probs, axis=1)
    s2_true  = df_s2['target'].values

    n_s1 = len(df_s1)                             # coverage denominator
    ias_pos = np.where(s1_true == 1)[0]
    assert len(ias_pos) == len(df_s2), \
        f"S1 AAC ({len(ias_pos)}) != S2 rows ({len(df_s2)}) — alignment error"

    records = []
    for s1_t in s1_thresholds:
        s1_pred_pos = s1_probs >= s1_t
        reaches_s2  = s1_pred_pos[ias_pos]        # true-AAC & passing S1
        for s2_t in s2_thresholds:
            confident   = s2_conf >= s2_t
            covered     = reaches_s2 & confident  # passing S1 + sufficient S2 conf
            n_cov       = int(covered.sum())
            overall_acc = (s2_pred[covered] == s2_true[covered]).mean() if n_cov > 0 else np.nan
            coverage    = n_cov / n_s1
            records.append({
                's1_threshold': round(float(s1_t), 2),
                's2_threshold': round(float(s2_t), 2),
                'overall_accuracy': overall_acc,
                'coverage': coverage,
                'n_covered': n_cov,
                'n_s1': n_s1,
            })
    return pd.DataFrame(records)


# ── Grid + joint sweep per scenario (TEST set) ────────────────────────────────
s1_grid = np.arange(0.30, 0.71, 0.05)
s2_grid = np.arange(0.35, 0.91, 0.05)

joint_results = {}
for scenario in SCENARIOS:
    joint_results[scenario] = joint_sweep(
        data[f'{scenario}_S1_test'],
        data[f'{scenario}_S2_test'],
        s1_grid, s2_grid,
    )
    print(f"✓ joint_results['{scenario}'] — {len(joint_results[scenario])} rows "
          f"(grid {len(s1_grid)}×{len(s2_grid)})")

# Operating point quick check (s2=0.35)
for scn, s1t in [('CBC_Only', 0.50), ('CBC_BIO', 0.44)]:
    jr = joint_results[scn]
    row = jr[(jr['s1_threshold']==round(s1t,2)) & (jr['s2_threshold']==0.35)]
    if len(row):
        print(f"  {scn} @ (s1={s1t}, s2=0.35): "
              f"acc={row['overall_accuracy'].iloc[0]:.4f}  cov={row['coverage'].iloc[0]:.4f}")

## Cell 9 — 2C: Joint Optimization Heatmap

In [ ]:
# ============================================================
# Cell 9 — Joint Optimization Heatmap
# ============================================================
from matplotlib.colors import LinearSegmentedColormap

# Custom colormaps from the project palette
cmap_acc = LinearSegmentedColormap.from_list('acc',
    [PALETTE['highlight'], PALETTE['accent2'], PALETTE['accent1']], N=256)
cmap_cov = LinearSegmentedColormap.from_list('cov',
    [PALETTE['base2'], PALETTE['accent3'], PALETTE['base1']], N=256)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for col, scenario in enumerate(SCENARIOS):
    jr = joint_results[scenario]

    for row, (metric, title, cmap) in enumerate([
        ('overall_accuracy', 'Overall Accuracy', cmap_acc),
        ('coverage', 'Coverage', cmap_cov)
    ]):
        ax = axes[row, col]

        pivot = jr.pivot(index='s2_threshold', columns='s1_threshold', values=metric)
        pivot = pivot.sort_index(ascending=False)

        im = ax.imshow(pivot.values, aspect='auto', cmap=cmap,
                       vmin=pivot.values.min(), vmax=pivot.values.max())

        s1_vals = pivot.columns.values
        s2_vals = pivot.index.values
        ax.set_xticks(range(len(s1_vals)))
        ax.set_xticklabels([f'{v:.2f}' for v in s1_vals], fontsize=7, rotation=45)
        ax.set_yticks(range(len(s2_vals)))
        ax.set_yticklabels([f'{v:.2f}' for v in s2_vals], fontsize=7)

        ax.set_xlabel('S1 Threshold', fontsize=9)
        ax.set_ylabel('S2 Confidence', fontsize=9)
        ax.set_title(f'{scenario} — {title}', fontsize=10, fontweight='bold')

        # Cell values — always black, on a semi-transparent white background
        for i in range(len(s2_vals)):
            for j in range(len(s1_vals)):
                val = pivot.values[i, j]
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=6, color='black',
                        bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                                  alpha=0.6, edgecolor='none'))

        plt.colorbar(im, ax=ax, shrink=0.8)

        # Thesis reference point
        thesis_s1 = 0.53 if scenario == 'CBC_Only' else 0.44
        thesis_s2 = 0.30
        s1_idx = np.argmin(np.abs(s1_vals - thesis_s1))
        s2_idx = np.argmin(np.abs(s2_vals - thesis_s2))
        ax.plot(s1_idx, s2_idx, '*', color=PALETTE['highlight'], markersize=14,
                markeredgecolor='black', markeredgewidth=0.8)

fig.suptitle('Joint Optimization — S1 Threshold × S2 Confidence',
             fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m2_thresholds', 'plots'), 's2_joint_heatmap')
plt.show()

## Cell 10 — Operating Point Selection + Test Validation + Config

In [ ]:
# ============================================================
# Cell 10 — Operating Point Selection + Test Validation + Config
# ============================================================
import json

# ── Selected operating points ──
# CBC_Only: S1=0.50 (plateau), S2 zones: LOW<0.35, HIGH≥0.82
# CBC_BIO:  S1=0.44 (thesis), S2 zones: LOW<0.35, HIGH≥0.60

operating_points = {
    'CBC_Only': {
        's1_threshold': 0.50,
        's1_method': 'F1-optimal (plateau — equivalent over 0.26-0.55)',
        's2_zone_low': 0.35,
        's2_zone_high': 0.82,
        's2_method': 'Accuracy≥0.85-based zone'
    },
    'CBC_BIO': {
        's1_threshold': 0.44,
        's1_method': 'Clinically safe (Sens≥0.95, thesis reference)',
        's2_zone_low': 0.35,
        's2_zone_high': 0.60,
        's2_method': 'Accuracy≥0.85-based zone'
    }
}

# ── Test set validation ──
print("═══ Test Set Validation — Selected Operating Points ═══\n")

test_results = []
for scenario in SCENARIOS:
    op = operating_points[scenario]

    # --- Stage 1 Test ---
    df_s1_test = data[f'{scenario}_S1_test']
    s1_metrics = evaluate_threshold(df_s1_test, op['s1_threshold'])

    # --- Stage 2 Test (zone distribution) ---
    df_s2_test = data[f'{scenario}_S2_test']
    zdf_test = assign_zones(df_s2_test, {'low': op['s2_zone_low'], 'high': op['s2_zone_high']})

    # --- Joint Test ---
    jr_test = joint_sweep(
        df_s1_test, df_s2_test,
        s1_thresholds=[op['s1_threshold']],
        s2_thresholds=[op['s2_zone_low']]  # minimum confidence = zone LOW boundary
    )

    print(f"{'═'*60}")
    print(f"  {scenario}")
    print(f"  S1 threshold = {op['s1_threshold']} | S2 zones: LOW<{op['s2_zone_low']}, HIGH≥{op['s2_zone_high']}")
    print(f"{'═'*60}")

    print(f"\n  Stage 1 (Test, n={len(df_s1_test)}):")
    print(f"    Sensitivity: {s1_metrics['sensitivity']:.3f}")
    print(f"    Specificity: {s1_metrics['specificity']:.3f}")
    print(f"    F1:          {s1_metrics['f1']:.3f}")
    print(f"    Accuracy:    {s1_metrics['accuracy']:.3f}")

    print(f"\n  Stage 2 Zone Distribution (Test, n={len(df_s2_test)}):")
    for zone in ['HIGH', 'MEDIUM', 'LOW']:
        mask = zdf_test['zone'] == zone
        n = mask.sum()
        pct = n / len(zdf_test) * 100
        acc = zdf_test.loc[mask, 'correct'].mean() if n > 0 else 0
        print(f"    {zone:<8} n={n:>4} ({pct:>5.1f}%)  acc={acc:.3f}")

    if len(jr_test) > 0:
        jt = jr_test.iloc[0]
        print(f"\n  Joint (Test):")
        print(f"    Overall Accuracy: {jt['overall_accuracy']:.3f}")
        print(f"    Coverage:         {jt['coverage']:.3f}")

    # Thesis comparison
    thesis_s1 = 0.53 if scenario == 'CBC_Only' else 0.44
    thesis_metrics = evaluate_threshold(df_s1_test, thesis_s1)
    print(f"\n  Thesis Comparison (S1 threshold):")
    print(f"    Thesis ({thesis_s1}):   F1={thesis_metrics['f1']:.3f}, Sens={thesis_metrics['sensitivity']:.3f}")
    print(f"    Selected ({op['s1_threshold']}): F1={s1_metrics['f1']:.3f}, Sens={s1_metrics['sensitivity']:.3f}")

    test_results.append({
        'scenario': scenario,
        's1_threshold': op['s1_threshold'],
        's1_sensitivity': s1_metrics['sensitivity'],
        's1_specificity': s1_metrics['specificity'],
        's1_f1': s1_metrics['f1'],
        's1_accuracy': s1_metrics['accuracy'],
        's2_zone_low': op['s2_zone_low'],
        's2_zone_high': op['s2_zone_high'],
        's2_high_pct': (zdf_test['zone'] == 'HIGH').mean(),
        's2_high_acc': zdf_test.loc[zdf_test['zone'] == 'HIGH', 'correct'].mean() if (zdf_test['zone'] == 'HIGH').sum() > 0 else 0,
        'joint_accuracy': jt['overall_accuracy'] if len(jr_test) > 0 else None,
        'joint_coverage': jt['coverage'] if len(jr_test) > 0 else None,
    })
    print()

# ── Operating Points Excel ──
df_op = pd.DataFrame(test_results)
op_path = PATHS.module_dir('m2_thresholds', 'metrics') / 'operating_points.xlsx'
df_op.round(4).to_excel(op_path, index=False)
print(f"✓ Saved: {op_path}")

# ── Threshold Config JSON ──
config = {
    'version': 'M2_v1',
    'created': pd.Timestamp.now().isoformat(),
    'scenarios': {}
}
for scenario in SCENARIOS:
    op = operating_points[scenario]
    config['scenarios'][scenario] = {
        'stage1': {
            'threshold': op['s1_threshold'],
            'method': op['s1_method'],
            'positive_class': 'IAS',
            'probability_column': 'prob_IAS_cal'
        },
        'stage2': {
            'zone_low': op['s2_zone_low'],
            'zone_high': op['s2_zone_high'],
            'method': op['s2_method'],
            'probability_columns': S2_CAL_COLS,
            'class_order': S2_CLASSES,
            'zones': {
                'HIGH': f'confidence >= {op["s2_zone_high"]}',
                'MEDIUM': f'{op["s2_zone_low"]} <= confidence < {op["s2_zone_high"]}',
                'LOW': f'confidence < {op["s2_zone_low"]}'
            }
        }
    }

config_path = PATHS.module_dir('m2_thresholds', 'configs') / 'threshold_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print(f"✓ Saved: {config_path}")

## Cell 11 — src/m2_thresholds.py module

In [ ]:
# ============================================================
# Cell 11 — src/m2_thresholds.py module
# ============================================================

import importlib.util
m2_code = '''"""
M2 — Threshold Optimization & Confidence Zones
================================================
Functions for runtime threshold and zone assignment in the CDS pipeline.

Usage:
    from m2_thresholds import load_threshold_config, apply_stage1_threshold, assign_confidence_zone

Last updated: Chat 2
"""

import json
import numpy as np
import pandas as pd
from pathlib import Path

# ── Constants ──
S2_CAL_COLS = ['prob_DEA_cal', 'prob_HA_cal', 'prob_HGB_HTZ_cal', 'prob_Normal_cal']
S2_CLASSES  = ['DEA', 'HA', 'HGB_HTZ', 'Normal']


def load_threshold_config(paths):
    """Load the threshold_config.json file."""
    config_path = paths.module_dir('m2_thresholds', 'configs') / 'threshold_config.json'
    with open(config_path, 'r') as f:
        return json.load(f)


def apply_stage1_threshold(df, threshold, prob_col='prob_IAS_cal'):
    """
    Apply the Stage 1 binary threshold.

    Returns:
        pd.Series: 'IAS' or 'DAS' string labels [internal codes]
    """
    return pd.Series(
        np.where(df[prob_col].values >= threshold, 'IAS', 'DAS'),
        index=df.index, name='s1_prediction'
    )


def assign_confidence_zone(df, zone_low, zone_high, prob_cols=None):
    """
    Stage 2 confidence zone assignment.

    Args:
        df: DataFrame containing Stage 2 probabilities
        zone_low: LOW/MEDIUM boundary
        zone_high: MEDIUM/HIGH boundary
        prob_cols: Probability columns (default: S2_CAL_COLS)

    Returns:
        pd.DataFrame: pred_class, confidence, zone, pred_label columns
    """
    if prob_cols is None:
        prob_cols = S2_CAL_COLS

    probs = df[prob_cols].values
    pred_idx = np.argmax(probs, axis=1)
    confidence = np.max(probs, axis=1)

    zones = np.where(confidence < zone_low, 'LOW',
            np.where(confidence >= zone_high, 'HIGH', 'MEDIUM'))

    pred_labels = [S2_CLASSES[i] for i in pred_idx]

    return pd.DataFrame({
        'pred_class': pred_idx,
        'pred_label': pred_labels,
        'confidence': confidence,
        'zone': zones
    }, index=df.index)


def get_operating_point(config, scenario):
    """Return the operating point for a given scenario."""
    sc = config['scenarios'][scenario]
    return {
        's1_threshold': sc['stage1']['threshold'],
        's2_zone_low':  sc['stage2']['zone_low'],
        's2_zone_high': sc['stage2']['zone_high']
    }


def apply_full_pipeline(df_s1, df_s2, config, scenario):
    """
    Full CDS pipeline: S1 threshold + S2 zone assignment.

    Returns:
        s1_pred: Stage 1 predictions
        s2_result: Stage 2 zone assignments (only for those passing as IAS)
    """
    op = get_operating_point(config, scenario)

    # Stage 1
    s1_pred = apply_stage1_threshold(df_s1, op['s1_threshold'])

    # Stage 2 (only those passing as IAS)
    s2_result = assign_confidence_zone(df_s2, op['s2_zone_low'], op['s2_zone_high'])

    return s1_pred, s2_result
'''

# Write to file
m2_path = PATHS.src / 'm2_thresholds.py'
with open(m2_path, 'w') as f:
    f.write(m2_code)
print(f"✓ Saved: {m2_path}")

# ── Import test ──
spec = importlib.util.spec_from_file_location("m2_thresholds", str(m2_path))
m2_mod = importlib.util.module_from_spec(spec)
sys.modules['m2_thresholds'] = m2_mod
spec.loader.exec_module(m2_mod)

from m2_thresholds import load_threshold_config, apply_stage1_threshold, assign_confidence_zone, get_operating_point

# Quick test
config = load_threshold_config(PATHS)
for scenario in SCENARIOS:
    op = get_operating_point(config, scenario)
    print(f"{scenario}: S1={op['s1_threshold']}, S2 zones=[{op['s2_zone_low']}, {op['s2_zone_high']}]")

    # Example application
    s1_pred = apply_stage1_threshold(data[f'{scenario}_S1_test'], op['s1_threshold'])
    s2_zones = assign_confidence_zone(data[f'{scenario}_S2_test'], op['s2_zone_low'], op['s2_zone_high'])
    print(f"  S1: {s1_pred.value_counts().to_dict()}")
    print(f"  S2 zones: {s2_zones['zone'].value_counts().to_dict()}")